In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install ../databricks_helpers
dbutils.library.restartPython()

In [0]:
from databricks_helpers.databricks_helper import DatabricksHelpers
dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS raw_data;
SELECT current_catalog(), current_schema();

In [0]:
%sql
select count(*) from bronze_flight_data;

In [0]:
%sql
select * From bronze_weather_data  limit 5;

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# 1. Define the target Silver table
silver_table_name = "silver_flight_weather"
checkpoint_path = f"{dbh.checkpoints_directory()}/silver_flight_weather"

# 2. Read the Bronze Flights as a Stream
bronze_flights_stream = spark.readStream.table("bronze_flight_data")

# 3. Define the Upsert Logic (The foreachBatch function)
def process_and_upsert_silver(microBatchDF, batchId):
    # a. Filter out corrupted data (where rescued data is NOT null)
    clean_flights = microBatchDF.filter(col("_rescued_data").isNull())
    
    # b. Read the Bronze Weather as a static table for this micro-batch
    # (We assume the weather bronze job runs before this silver job)
    clean_weather = (spark.read.table("bronze_weather_data")
                     .filter(col("_rescued_data").isNull()))
    
    # c. Join Flights with Weather
    # (Assuming weather data has an 'airport_code' and 'time' column as discussed)
    silver_batch_df = clean_flights.join(
        clean_weather,
        (clean_flights.FL_DATE == clean_weather.time) & 
        (clean_flights.ORIGIN == clean_weather.iata_code),
        "left"
    ).select(
        col("FL_DATE").cast("date").alias("Flight_Date"),
        col("OP_CARRIER_FL_NUM").alias("Flight_Number"),
        col("OP_UNIQUE_CARRIER").alias("Carrier"),
        col("ORIGIN").alias("Origin"),
        col("DEST").alias("Destination"),
        col("DEP_DELAY").cast("int").alias("Departure_Delay"),
        col("CANCELLED").cast("int").alias("Is_Cancelled"),
        col("temp").alias("Avg_Temp_C"),
        col("snwd").alias("Snow_Depth_cm"),
        col("rhum").alias("Relative_Humidity"),
        col("wspd").alias("Avg_Wind_Speed_kmph"),
        col("pres").alias("Avg_Air_Pressure_hpa"),
        col("prcp").alias("Total_Precipitation_mm")
    )
    
    # d. Handle the Upsert (MERGE)
    if not spark.catalog.tableExists(silver_table_name):
        # If it's the very first run, just save the table
        silver_batch_df.write.format("delta").saveAsTable(silver_table_name)
    else:
        # If the table exists, perform the MERGE to avoid duplicates
        silver_table = DeltaTable.forName(spark, silver_table_name)
        
        (silver_table.alias("target")
            .merge(
                silver_batch_df.alias("updates"),
                # The Composite Key: How we identify a unique flight
                """
                target.FL_DATE = updates.FL_DATE AND
                target.Origin = updates.Origin AND
                target.Flight_Number = updates.Flight_Number AND
                target.Carrier = updates.Carrier
                """
            )
            .whenMatchedUpdateAll()     # Update existing flights (e.g., late cancellations)
            .whenNotMatchedInsertAll()  # Insert brand new flights
            .execute()
        )

# 4. Start the Stream using foreachBatch
(bronze_flights_stream.writeStream
    .foreachBatch(process_and_upsert_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True) # Turn on, process new Bronze rows, turn off
    .start()
)

In [0]:
%sql
select * From silver_flight_weather;

In [0]:
%sql
drop table silver_flight_weather;